# Phase 1: Bonsai-8B ベースライン測定

**目的**: test_normal (168 タスク) でスキャフォールドなしの成績を測定

**所要時間見込み**: 168 タスク × 37.6 秒 ≒ 約 1.7 時間

## 1. 環境セットアップ

In [ ]:
# 依存インストール
!pip install -q "pydantic==1.10.26" "fastapi==0.110.3"
!pip install -q appworld huggingface_hub openai
!appworld install
!appworld download data

In [ ]:
import os
import json
import re
import subprocess
import time
import requests
from pathlib import Path
from datetime import datetime
from collections import Counter

In [ ]:
!nvidia-smi

## 2. Bonsai-8B 推論サーバー起動

In [ ]:
# モデルダウンロード
from huggingface_hub import hf_hub_download

MODEL_DIR = Path("/content/bonsai-8b-gguf")
MODEL_DIR.mkdir(exist_ok=True)

model_path = hf_hub_download(
    repo_id="prism-ml/Bonsai-8B-gguf",
    filename="Bonsai-8B.gguf",
    local_dir=str(MODEL_DIR),
)
print(f"Model: {model_path}")

In [ ]:
# PrismML フォーク llama.cpp ビルド
!git clone https://github.com/PrismML-Eng/llama.cpp /content/llama-cpp-prism 2>/dev/null || echo "Already cloned"
%cd /content/llama-cpp-prism
!cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native 2>&1 | tail -3
!cmake --build build --config Release -j$(nproc) 2>&1 | tail -5
!ls -la build/bin/llama-server 2>/dev/null || echo "BUILD FAILED"
%cd /content

In [ ]:
# サーバー起動 (ctx=16384, ポート 8090)
!pkill -9 -f llama-server 2>/dev/null; sleep 3

server_proc = subprocess.Popen(
    [
        "/content/llama-cpp-prism/build/bin/llama-server",
        "-m", str(list(MODEL_DIR.glob("*.gguf"))[0]),
        "--host", "0.0.0.0",
        "--port", "8090",
        "-ngl", "99",
        "-c", "16384",
    ],
    stdout=open("/content/server.log", "w"),
    stderr=subprocess.STDOUT,
)

time.sleep(20)
!cat /content/server.log | tail -5

In [ ]:
# 動作確認
from openai import OpenAI

BONSAI_PORT = 8090
client = OpenAI(base_url=f"http://localhost:{BONSAI_PORT}/v1", api_key="dummy")

response = client.chat.completions.create(
    model="bonsai-8b",
    messages=[{"role": "user", "content": "What is 15 + 27?"}],
    temperature=0,
    max_tokens=200,
    seed=100,
)
print(f"Response: {response.choices[0].message.content}")
print("Server OK")

## 3. AppWorld 準備

In [ ]:
import appworld
appworld.update_root("/content")

from appworld import AppWorld, load_task_ids
from appworld.task import Task

test_task_ids = load_task_ids("test_normal")
print(f"test_normal tasks: {len(test_task_ids)}")

# 難易度別カウント
difficulties = Counter()
for tid in test_task_ids:
    task = Task.load(task_id=tid)
    difficulties[task.ground_truth.metadata["difficulty"]] += 1
for d in sorted(difficulties):
    print(f"  Difficulty {d}: {difficulties[d]} tasks")

## 4. エージェント定義

In [ ]:
def call_bonsai(messages: list[dict], max_tokens: int = 3000) -> str:
    """Bonsai LLM を OpenAI 互換 API 経由で呼び出す"""
    response = client.chat.completions.create(
        model="bonsai-8b",
        messages=messages,
        temperature=0,
        max_tokens=max_tokens,
        seed=100,
    )
    return response.choices[0].message.content


def extract_code(text: str) -> str | None:
    """LLM 出力から Python コードブロックを抽出"""
    pattern = r"```python\s*\n(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    if "apis." in text:
        return text.strip()
    return None


SYSTEM_PROMPT = """You are an AI assistant that solves tasks by writing Python code.
You have access to APIs via the `apis` object. Write code to accomplish the task.
Always wrap your code in ```python ... ``` blocks.
Keep your code concise and focused on the task."""


def run_baseline_task(task_id: str, max_steps: int = 40) -> dict:
    """ベースライン: 単一 LLM 呼び出しで 1 タスクを実行 (max_steps=40 は論文準拠)"""
    result = {
        "task_id": task_id,
        "success": False,
        "steps": 0,
        "turns": [],
        "wall_time": 0,
        "error": None,
    }
    t0 = time.time()
    try:
        with AppWorld(task_id=task_id, experiment_name="phase1_baseline") as world:
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": (
                    f"Task: {world.task.instruction}\n\n"
                    f"Supervisor: {world.task.supervisor}\n\n"
                    f"Available apps: {list(world.task.app_descriptions.keys())}"
                )},
            ]
            for step in range(max_steps):
                turn_t0 = time.time()
                try:
                    llm_output = call_bonsai(messages)
                except Exception as llm_err:
                    # コンテキスト超過等の LLM エラー → タスク終了
                    result["error"] = str(llm_err)
                    result["steps"] = step
                    break
                turn_time = time.time() - turn_t0
                code = extract_code(llm_output)
                turn_log = {
                    "step": step,
                    "prompt_messages": len(messages),
                    "llm_output": llm_output,
                    "code_extracted": code is not None,
                    "turn_time": turn_time,
                }
                if code is None:
                    turn_log["observation"] = "ERROR: No code block found in LLM output"
                    messages.append({"role": "assistant", "content": llm_output})
                    messages.append({"role": "user", "content": turn_log["observation"]})
                    result["turns"].append(turn_log)
                    continue
                try:
                    output = world.execute(code)
                    turn_log["observation"] = output
                except Exception as e:
                    output = f"Execution error: {e}"
                    turn_log["observation"] = output
                messages.append({"role": "assistant", "content": llm_output})
                messages.append({"role": "user", "content": f"Output:\n```\n{output}\n```"})
                result["turns"].append(turn_log)
                result["steps"] = step + 1
                if world.task_completed():
                    result["success"] = True
                    break
    except Exception as e:
        result["error"] = str(e)
    result["wall_time"] = time.time() - t0
    return result

## 5. ベースライン評価 (test_normal 全タスク)

In [ ]:
RESULTS_DIR = Path("/content/phase1_results")
RESULTS_DIR.mkdir(exist_ok=True)

checkpoint_file = RESULTS_DIR / "phase1_baseline.jsonl"

# チェックポイント読み込み
all_results = []
completed_ids = set()
if checkpoint_file.exists():
    with open(checkpoint_file) as f:
        for line in f:
            r = json.loads(line)
            completed_ids.add(r["task_id"])
            all_results.append(r)
    print(f"Resuming: {len(completed_ids)}/{len(test_task_ids)} tasks already completed")

start_time = time.time()

for i, task_id in enumerate(test_task_ids):
    if task_id in completed_ids:
        continue

    elapsed_total = time.time() - start_time
    remaining = len(test_task_ids) - len(completed_ids) - (i - len(completed_ids))
    if len(all_results) > len(completed_ids):
        avg_time = elapsed_total / (len(all_results) - len(completed_ids))
        eta = avg_time * remaining
        eta_str = f", ETA: {eta/60:.0f}min"
    else:
        eta_str = ""

    print(f"[{i+1}/{len(test_task_ids)}] {task_id}{eta_str}")
    result = run_baseline_task(task_id, max_steps=40)
    all_results.append(result)
    completed_ids.add(task_id)

    # チェックポイント保存
    with open(checkpoint_file, "a") as f:
        f.write(json.dumps(result, ensure_ascii=False, default=str) + "\n")

    status = "OK" if result["success"] else "FAIL"
    err = f" [{result['error'][:60]}]" if result["error"] else ""
    print(f"  {status} steps={result['steps']} time={result['wall_time']:.1f}s{err}")

print(f"\nTotal time: {(time.time() - start_time)/60:.1f} min")

## 6. 結果分析

In [ ]:
# --- 全体サマリ ---
print("=" * 60)
print("PHASE 1 BASELINE RESULTS")
print("=" * 60)

successes = sum(1 for r in all_results if r["success"])
total = len(all_results)
errors = sum(1 for r in all_results if r["error"])
times = [r["wall_time"] for r in all_results]
steps = [r["steps"] for r in all_results]

print(f"Tasks: {total}")
print(f"Success: {successes}/{total} ({successes/total*100:.1f}%)")
print(f"Errors (context overflow etc): {errors}/{total}")
print(f"Wall time: mean={sum(times)/len(times):.1f}s, total={sum(times)/60:.1f}min")
print(f"Steps: mean={sum(steps)/len(steps):.1f}")

In [ ]:
# --- 難易度別 ---
print("\n=== Results by Difficulty ===")
by_difficulty = {}
for r in all_results:
    task = Task.load(task_id=r["task_id"])
    d = task.ground_truth.metadata["difficulty"]
    if d not in by_difficulty:
        by_difficulty[d] = {"total": 0, "success": 0}
    by_difficulty[d]["total"] += 1
    if r["success"]:
        by_difficulty[d]["success"] += 1

for d in sorted(by_difficulty):
    info = by_difficulty[d]
    rate = info["success"] / info["total"] * 100 if info["total"] > 0 else 0
    print(f"Difficulty {d}: {info['success']}/{info['total']} ({rate:.1f}%)")

In [ ]:
# --- コード抽出率 ---
all_turns = [t for r in all_results for t in r["turns"]]
code_extracted = sum(1 for t in all_turns if t["code_extracted"])
print(f"\nCode extraction rate: {code_extracted}/{len(all_turns)} "
      f"({code_extracted/len(all_turns)*100:.1f}%)")

In [ ]:
# --- エラーパターン ---
print("\n=== Error Patterns ===")
error_types = Counter()
for r in all_results:
    if r["error"]:
        if "exceeds the available" in r["error"]:
            error_types["context_overflow"] += 1
        elif "Connection" in r["error"]:
            error_types["connection_error"] += 1
        else:
            error_types["other"] += 1

for etype, count in error_types.most_common():
    print(f"  {etype}: {count}")

## 7. Phase 1 レポート

In [ ]:
report = {
    "timestamp": datetime.now().isoformat(),
    "phase": "1-baseline",
    "inference_backend": "llama.cpp (PrismML fork)",
    "gpu": "NVIDIA A100-SXM4-40GB",
    "context_length": 16384,
    "dataset": "test_normal",
    "total_tasks": total,
    "successes": successes,
    "success_rate": successes / total if total > 0 else 0,
    "errors": errors,
    "mean_wall_time_seconds": sum(times) / len(times) if times else 0,
    "total_wall_time_minutes": sum(times) / 60 if times else 0,
    "mean_steps": sum(steps) / len(steps) if steps else 0,
    "code_extraction_rate": code_extracted / len(all_turns) if all_turns else 0,
    "by_difficulty": {
        str(d): {"total": info["total"], "success": info["success"]}
        for d, info in sorted(by_difficulty.items())
    },
    "error_patterns": dict(error_types),
}

report_file = RESULTS_DIR / "phase1_report.json"
with open(report_file, "w") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2, ensure_ascii=False))